# Model comparison: BERTopic with `paraphrase-multilingual-mpnet-base-v2`

Comparison run for the choice of the embedding model (paper: „Vergleich dreier Embedding-Modelle“). Identical to the final model in `02_bertopic` except for the embedding model: same sentences, same UMAP and HDBSCAN parameters, same random seed, same stopword list.

**Embedding model**: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` (multilingual, 768 dimensions, max. 128 word pieces)

**Input** (from `02_bertopic`)
- `../02_bertopic/input/lesefassung.txt` – plain text; paragraphs are delimited by `///`, sentences by `###`
- `../02_bertopic/stopwords/stopwords_edited.txt` – edited French stopword list, used for the topic representations only

**Output**
- `models/embeddings_mpnet.npy` – sentence embeddings (not published)
- `models/mpnet.mm` – the fitted model (not published)
- `output/embedding_models/topic_info_with_weights_mpnet.json` – topics with top terms and c-TF-IDF weights
- `output/embedding_models/document_info_mpnet.json` – topic assignment per sentence

**Environment**: Google Colab, T4 GPU, package versions in `../02_bertopic/requirements.txt`. The published outputs were produced on 2026-09-21.

Run the notebook with this folder as working directory; all paths are relative.

## Setup

In [ ]:
# Install the pinned versions (uncomment if needed)
# %pip install -r ../02_bertopic/requirements.txt

# In Google Colab: mount Drive and change into the folder containing this notebook
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<path-to>/05_model_comparison

In [ ]:
from bertopic import BERTopic
import pandas as pd

# Prepare data

Splits the text into sentences at `###` and records the paragraph number of each sentence (paragraphs are split at `///`).

In [ ]:
import re

file_path = '../02_bertopic/input/lesefassung.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

def clean_text(sentence):
    # Replace non-breaking spaces and other non-standard spaces
    sentence = sentence.replace('\xa0', ' ').replace('\n', ' ')
    # Replace multiple spaces with a single space
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence

def process_text_for_bertopic(text):
    # Split the text into paragraphs using '///' as the delimiter
    paragraphs = text.split('///')
    sentences = []
    sentence_paragraph_map = []

    # Enumerate the paragraphs and split sentences at '###'
    for idx, paragraph in enumerate(paragraphs):
        # Trim whitespace which might be left from splitting
        trimmed_paragraph = paragraph.strip()
        if trimmed_paragraph:  # Ensure the paragraph is not empty
            # Split the paragraph into sentences using '###'
            paragraph_sentences = [s.strip() for s in trimmed_paragraph.split('###') if s.strip()]
            # Clean and extend the sentences to the main list
            cleaned_sentences = [clean_text(sentence) for sentence in paragraph_sentences]
            sentences.extend(cleaned_sentences)
            # Store the paragraph number for each sentence for later analysis
            sentence_paragraph_map.extend([idx + 1] * len(cleaned_sentences))

    return sentences, sentence_paragraph_map

# Process the text for BERTopic
sentences, sentence_paragraph_map = process_text_for_bertopic(text)

print(f"{len(sentences)} sentences")


# Create topics

In [ ]:
import os
import random
import numpy as np
import torch
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer

# -----------------------------------------------------
# 1. Fix all random seeds
# -----------------------------------------------------
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# -----------------------------------------------------
# 2. Embedding model — compute once, save, reuse
# -----------------------------------------------------
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("dim:", embedding_model.get_sentence_embedding_dimension())
print("max_seq_length:", embedding_model.max_seq_length)

embeddings = embedding_model.encode(sentences, show_progress_bar=True, batch_size=64)
os.makedirs("./models", exist_ok=True)
np.save("./models/embeddings_mpnet.npy", embeddings)

# -----------------------------------------------------
# 3. Define deterministic subcomponents
# -----------------------------------------------------
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# -----------------------------------------------------
# 4. Define BERTopic with the chosen embedding model
# -----------------------------------------------------
topic_model = BERTopic(
    verbose=True,
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model
)

# -----------------------------------------------------
# 5. Fit with the precomputed embeddings
# -----------------------------------------------------
topics, probs = topic_model.fit_transform(sentences, embeddings)

# Save the model

The model is saved right after fitting, i.e. before the stopword update below.

In [ ]:
topic_model.save("./models/mpnet.mm")

In [ ]:
topic_model = BERTopic.load("./models/mpnet.mm")

# Inspect

Shows the embedding model used.

In [ ]:
print(topic_model.embedding_model.embedding_model)
print(topic_model.embedding_model.embedding_model._modules['0'].auto_model.config._name_or_path)

# Finetuning the model

Recomputes the topic representations with the edited stopword list. This only changes the terms describing each topic, not the assignment of sentences to topics.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Load the edited French stopword list

with open('../02_bertopic/stopwords/stopwords_edited.txt', 'r', encoding='utf-8') as file:
  stop_words = [word.strip() for word in file.readlines()]

vectorizer_model = CountVectorizer(stop_words=stop_words)
topic_model.update_topics(sentences, vectorizer_model=vectorizer_model)

# Investigate the topics

In [ ]:
topic_model.get_topic_info()

In [ ]:
import json

# Retrieve the topic info DataFrame
topic_info_df = topic_model.get_topic_info()

# Retrieve weights for each topic
topics_with_weights = {}
for topic_id in topic_info_df["Topic"]:
    if topic_id != -1:  # Skip the catch-all topic (-1) if not needed
        # Retrieve terms and their weights for the topic
        topics_with_weights[topic_id] = topic_model.get_topic(topic_id)

# Combine the topic info with the weights
topic_info_with_weights = []
for _, row in topic_info_df.iterrows():
    topic_id = row["Topic"]
    row_data = row.to_dict()
    if topic_id in topics_with_weights:
        # Add terms and weights for the topic
        row_data["Terms_with_Weights"] = topics_with_weights[topic_id]
    topic_info_with_weights.append(row_data)

# Convert the combined data to a JSON string
topic_info_with_weights_json = json.dumps(
    topic_info_with_weights, ensure_ascii=False, indent=4
)

# Specify the output file path
output_file = './output/embedding_models/topic_info_with_weights_mpnet.json'

# Save the JSON data to a file
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(topic_info_with_weights_json)

print(f"Topic info with weights has been saved to {output_file}")

# Topic labels and document info

In [ ]:
topic_labels = topic_model.generate_topic_labels(nr_words=20, topic_prefix=False, separator=" - ")
topic_model.set_topic_labels(topic_labels)
# topic_model.set_topic_labels({2: "Lineages"})

In [ ]:
# Retrieve the document info DataFrame based on sentences
document_info_df = topic_model.get_document_info(sentences)

# Convert the DataFrame to a JSON string
document_info_json = document_info_df.to_json(orient='records', force_ascii=False, indent=4)

# Specify the output file path
output_file = './output/embedding_models/document_info_mpnet.json'

# Save the JSON data to a file
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(document_info_json)

print(f"Document info has been saved to {output_file}")